In [19]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pl.Config.set_tbl_rows(100)

DATA_PATH = r"E:\Databard\DLD\dld-clean.parquet"

df = pl.read_parquet(DATA_PATH)

print(f"Rows: {df.height:,} | Columns: {df.width}")
print(df.schema)

Rows: 566,235 | Columns: 19
Schema({'instance_date': Date, 'area_name_en': String, 'building_name_en': String, 'project_name_en': String, 'master_project_en': String, 'nearest_landmark_en': String, 'nearest_metro_en': String, 'nearest_mall_en': String, 'has_parking': Int64, 'procedure_area': Float64, 'actual_worth': Float64, 'meter_sale_price': Float64, 'year': Int32, 'month': Int8, 'sale_type': String, 'property_cat': String, 'district': String, 'neighborhood': String, 'bedrooms': Int8})


## Dubai Land Department — Residential Sales Analysis

This notebook picks up where the EDA cleaning pipeline left off. We load the clean parquet snapshot and move straight into analysis: market-level KPIs, volume and price trends over time, the off-plan vs ready split, geographic breakdowns, and a deep dive into the bedroom segment most interesting from an investment perspective.

All price metrics use **median** throughout, not mean. The Dubai market contains ultra-luxury transactions that would distort any mean-based calculation. Median gives a more accurate picture of what a typical buyer actually paid.

## Section 1: Market Overview KPIs

Before any charts, we establish the headline numbers. Total transaction count, total market value transacted, median price per sqm, and median deal size give us the scale of the market we are analysing at a glance.

These are the numbers you would lead with in any executive summary or portfolio presentation. They anchor every subsequent chart in real scale.

In [20]:
total_transactions = df.height
total_value        = df["actual_worth"].sum()
median_sqm_price   = df["meter_sale_price"].median()
median_deal_size   = df["actual_worth"].median()

print("=" * 45)
print("  DUBAI RESIDENTIAL MARKET  |  2021–2026")
print("=" * 45)
print(f"  Total Transactions  : {total_transactions:>12,.0f}")
print(f"  Total Value (AED)   : {total_value:>12,.0f}")
print(f"  Total Value (B AED) : {total_value/1e9:>12.1f}")
print(f"  Median AED / sqm    : {median_sqm_price:>12,.0f}")
print(f"  Median Deal Size    : {median_deal_size:>12,.0f}")
print("=" * 45)

  DUBAI RESIDENTIAL MARKET  |  2021–2026
  Total Transactions  :      566,235
  Total Value (AED)   : 1,159,534,578,705
  Total Value (B AED) :       1159.5
  Median AED / sqm    :       16,825
  Median Deal Size    :    1,395,882


## Section 2: Transaction Volume Trends

We look at how many deals happened each year and each month. Volume is as important as price: a market can show rising prices while volume collapses (illiquid, fewer willing sellers) or rising prices alongside rising volume (genuine demand expansion).

The yearly bar chart gives the macro view. The monthly line chart shows seasonality and turning points within each year that annual aggregation hides.

In [21]:
# Yearly volume
yearly = (
    df.group_by("year")
      .agg(pl.len().alias("transactions"))
      .sort("year")
)

fig = px.bar(
    yearly,
    x="year", y="transactions",
    title="Transaction Volume by Year",
    labels={"transactions": "Transactions", "year": "Year"},
    text_auto=True,
)
fig.update_traces(textposition="outside")
fig.show()

# Monthly volume heatmap
monthly = (
    df.group_by(["year", "month"])
      .agg(pl.len().alias("transactions"))
      .sort(["year", "month"])
)

fig2 = px.density_heatmap(
    monthly,
    x="month", y="year",
    z="transactions",
    title="Transaction Volume Heatmap (Year × Month)",
    labels={"month": "Month", "year": "Year", "transactions": "Transactions"},
    color_continuous_scale="Blues",
)
fig2.show()

## Section 3: Price Trends Over Time

We track median price per sqm month by month across the full dataset period. Price per sqm is the only fair comparison across unit sizes and neighborhoods: a 3-bedroom flat and a studio are incomparable on total deal value, but comparable on AED per sqm.

Monthly granularity lets us spot acceleration, plateaus, and any softening periods that an annual view would smooth over.

In [22]:
price_trend = (
    df.group_by(["year", "month"])
      .agg(pl.median("meter_sale_price").alias("median_sqm_price"))
      .sort(["year", "month"])
      .with_columns(
          pl.date(pl.col("year"), pl.col("month"), pl.lit(1)).alias("date")
      )
)

fig = px.line(
    price_trend,
    x="date", y="median_sqm_price",
    title="Median Price per sqm Over Time (AED)",
    labels={"median_sqm_price": "Median AED / sqm", "date": ""},
    markers=True,
)
fig.update_layout(hovermode="x unified")
fig.show()

## Section 4: Off-Plan vs Ready Split

We quantify the off-plan vs ready balance both by transaction count and by total value. These two views can diverge: off-plan units are typically cheaper per deal (payment plans, developer incentives), so their share of total value may be lower than their share of transaction count even when they dominate by volume.

Understanding this split tells us how much of the market is speculative (betting on future delivery) vs transactional (buying existing stock), which has direct implications for supply pipeline risk and price stability.

### Off-Plan vs Ready

In [23]:
# Volume split
split_vol = df["sale_type"].value_counts().sort("count", descending=True)
fig = px.pie(
    split_vol,
    names="sale_type", values="count",
    title="Transaction Volume: Off-Plan vs Ready",
    hole=0.4,
)
fig.show()

# Price comparison over time
split_trend = (
    df.group_by(["year", "month", "sale_type"])
      .agg(pl.median("meter_sale_price").alias("median_sqm_price"))
      .sort(["year", "month"])
      .with_columns(
          pl.date(pl.col("year"), pl.col("month"), pl.lit(1)).alias("date")
      )
)

fig2 = px.line(
    split_trend,
    x="date", y="median_sqm_price", color="sale_type",
    title="Median Price per sqm: Off-Plan vs Ready",
    labels={"median_sqm_price": "Median AED / sqm", "date": "", "sale_type": "Type"},
    markers=True,
)
fig2.update_layout(hovermode="x unified")
fig2.show()

## Section 5: Geographic Breakdown

We rank neighborhoods by two separate metrics: transaction volume and median price per sqm. These two rankings rarely match.

The highest-volume areas are typically affordable mid-market communities with deep buyer pools. The highest-price areas are typically low-volume luxury pockets. Showing both side by side reveals the structural two-speed nature of the market: where the money is concentrated vs where the deals happen.

In [24]:
# Top 20 by volume
top_vol = (
    df.group_by("neighborhood")
      .agg(pl.len().alias("transactions"))
      .sort("transactions", descending=True)
      .head(20)
)

fig = px.bar(
    top_vol,
    x="transactions", y="neighborhood",
    orientation="h",
    title="Top 20 Neighborhoods by Transaction Volume",
    labels={"transactions": "Transactions", "neighborhood": ""},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

# Top 20 by median price per sqm (min 100 transactions to exclude noise)
top_price = (
    df.group_by("neighborhood")
      .agg([
          pl.median("meter_sale_price").alias("median_sqm_price"),
          pl.len().alias("transactions"),
      ])
      .filter(pl.col("transactions") >= 100)
      .sort("median_sqm_price", descending=True)
      .head(20)
)

fig2 = px.bar(
    top_price,
    x="median_sqm_price", y="neighborhood",
    orientation="h",
    title="Top 20 Neighborhoods by Median Price per sqm (AED)",
    labels={"median_sqm_price": "Median AED / sqm", "neighborhood": ""},
)
fig2.update_layout(yaxis={"categoryorder": "total ascending"})
fig2.show()

In [25]:
bedroom_stats = (
    df.filter(pl.col("year") == 2025)
      .group_by("bedrooms")
      .agg([
          pl.median("actual_worth").alias("median_price"),
          pl.len().alias("transactions"),
      ])
      .sort("bedrooms")
)

labels         = ["Studio" if b == 0 else f"{b} B/R" for b in bedroom_stats["bedrooms"].to_list()]
median_prices  = [f"AED {p:,.0f}" for p in bedroom_stats["median_price"].to_list()]

bedroom_stats = bedroom_stats.with_columns(pl.Series("label", labels))

fig = px.bar(
    bedroom_stats,
    x="label", y="transactions",
    text=median_prices,
    title="Units Sold by Bedroom Count — 2025 (Median Price Labelled)",
    labels={"transactions": "Units Sold", "label": "Bedrooms"},
)
fig.update_traces(textposition="outside")
fig.show()

## Section 6: Bedroom Segment Analysis (2025)

We restrict to 2025 data here. Prices from 2021 or 2022 are not relevant for understanding what different bedroom counts cost today.

For each bedroom count we calculate the median sale price and the number of transactions. This gives us two dimensions simultaneously: what the typical buyer paid, and how liquid each segment is. A high-median, low-volume bedroom count (like 5 B/R) tells a very different story than a moderate-median, high-volume one (like 2 B/R).

In [26]:
bedroom_trend = (
    df.filter(pl.col("bedrooms") <= 5)
      .group_by(["year", "month", "bedrooms"])
      .agg(pl.median("actual_worth").alias("median_price"))
      .sort(["year", "month", "bedrooms"])
      .with_columns(
          pl.date(pl.col("year"), pl.col("month"), pl.lit(1)).alias("date")
      )
)

bedroom_trend = bedroom_trend.with_columns(
    pl.Series("label", [
        "Studio" if b == 0 else f"{b} B/R"
        for b in bedroom_trend["bedrooms"].to_list()
    ])
)

fig = px.line(
    bedroom_trend,
    x="date", y="median_price", color="label",
    title="Median Sale Price Over Time by Bedroom Count",
    labels={"median_price": "Median Price (AED)", "date": "", "label": "Bedrooms"},
    markers=False,
)
fig.update_layout(hovermode="x unified")
fig.show()

## Section 7: Price Trend by Bedroom Count Over Time

We plot median sale price per bedroom count as separate lines on one chart, restricted to Studio through 5 B/R. This shows whether all segments appreciated in parallel or whether certain bedroom counts pulled ahead of others.

If premium segments (4 B/R, 5 B/R) outpaced smaller units, it suggests luxury demand led the cycle. If studios and 1 B/R units grew fastest, it points to investor-driven affordability compression at the entry end of the market.

In [27]:
print(
    df.filter(pl.col("bedrooms") == 5)
      .group_by("year")
      .agg(pl.len().alias("sold"))
      .sort("year")
)

print(f"Total 5 B/R sold: {df.filter(pl.col('bedrooms') == 5).height:,}")

shape: (6, 2)
┌──────┬──────┐
│ year ┆ sold │
│ ---  ┆ ---  │
│ i32  ┆ u32  │
╞══════╪══════╡
│ 2021 ┆ 245  │
│ 2022 ┆ 421  │
│ 2023 ┆ 693  │
│ 2024 ┆ 917  │
│ 2025 ┆ 829  │
│ 2026 ┆ 640  │
└──────┴──────┘
Total 5 B/R sold: 3,745


## Section 8: 5-Bedroom Transaction Volume by Year

The 5 B/R segment is the most volatile and the most interesting. Before drawing any conclusions from its price trend, we need to know how many transactions actually happened each year.

A median based on 20 transactions is very different from a median based on 2,000. If volume is thin in certain years, the price line reflects the specific deals that happened rather than a true market level. This count check validates whether the 5 B/R trend line is statistically meaningful.

## Section 9: 5-Bedroom Villa Deep Dive (2025)

We filter to 5 B/R villas in 2025 only and print the full distribution: min, median, max, and a ranked table of deals. This is a focused look at the most illiquid, highest-variance segment in the dataset.

The goal is to understand not just what the typical 5 B/R villa costs today but how wide the range is, which neighborhoods they transact in, and what the outlier transactions look like. This segment is where ultra-luxury and mainstream villa markets overlap, and the spread between the cheapest and most expensive deal tells you how heterogeneous the product really is.

In [ ]:
villas_5br = df.filter(
    (pl.col("bedrooms") == 5) &
    (pl.col("property_cat") == "Villa") &
    (pl.col("year") == 2025)
)

print("=" * 50)
print("  5 BEDROOM VILLAS  |  2025")
print("=" * 50)
print(f"  Total Sold       : {villas_5br.height:>10,}")
print(f"  Min Price  (M)   : {villas_5br['actual_worth'].min()/1e6:>10.2f}M")
print(f"  Max Price  (M)   : {villas_5br['actual_worth'].max()/1e6:>10.2f}M")
print(f"  Median     (M)   : {villas_5br['actual_worth'].median()/1e6:>10.2f}M")
print(f"  Avg        (M)   : {villas_5br['actual_worth'].mean()/1e6:>10.2f}M")
print(f"  25th pct   (M)   : {villas_5br['actual_worth'].quantile(0.25)/1e6:>10.2f}M")
print(f"  75th pct   (M)   : {villas_5br['actual_worth'].quantile(0.75)/1e6:>10.2f}M")
print("=" * 50)

print("\nTop Neighborhoods:")
print(
    villas_5br.group_by("neighborhood")
      .agg([
          pl.len().alias("sold"),
          (pl.median("actual_worth") / 1e6).round(2).alias("median_M"),
          (pl.min("actual_worth")    / 1e6).round(2).alias("min_M"),
          (pl.max("actual_worth")    / 1e6).round(2).alias("max_M"),
      ])
      .sort("sold", descending=True)
      .head(10)
)